> **归档说明**：本 notebook 记录项目开发过程中的中间实验，保留用于复盘和审计，不作为最终展示入口。部分输入可能依赖本地生成但未上传 GitHub 的过程产物，例如 `outputs/predictions/`、`threshold_metrics` 或 `trial_results`。复现这些历史实验前，请先查看 `reports/notebook_reproducibility_audit.md` 中对应的再生成脚本说明。项目最终展示入口见 `notebooks/final/` 和 `README.md`。
>
> 该实验结果仅作为历史对照，不作为最终模型选择依据。


# 08 Validation-based Model Selection

本轮增强的目标是把 Day 6 / Day 7 的测试集回溯阈值分析，升级为更严谨的 `train_inner / valid / official test` 流程：在 `train_inner` 上训练模型，在 `valid` 上选择模型、缺失处理策略和阈值，最后在官方 test 上只评估一次。

本 notebook 不做 GridSearch，不做新的特征工程，不重新做风险分层，也不把结果写成生产环境最终阈值。

## 1. 为什么需要 validation set

Day 6 的低成本阈值来自官方 test 预测概率上的回溯敏感性分析。这对业务解释有帮助，但方法论上偏乐观，因为 test 同时参与了阈值选择和结果评估。

本轮增强使用官方 training set 内部划分出的 validation set 来选择阈值，official test 只保留为最终评估。这样可以观察原 Day 6 结果是否存在测试集回溯偏差。

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from scania_aps.config import get_config
from scania_aps.data.load_data import load_train_test_with_target
from scania_aps.data.split_data import split_train_valid
from scania_aps.models.validation_selection import run_validation_model_selection

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
cfg.validation_valid_size, cfg.validation_stratify, cfg.random_state

(0.2, True, 42)

## 2. 读取 official train/test 并划分 train_inner / valid

这里仍然使用官方 test set，不把 train/test 合并后重新划分。validation 只来自官方 training set。

In [2]:
train_df, test_df = load_train_test_with_target(cfg)
train_inner_df, valid_df = split_train_valid(train_df, cfg)

split_summary = pd.DataFrame(
    [
        {"dataset": "official_train", "row_count": len(train_df), "pos_count": int(train_df["target"].sum()), "pos_rate": train_df["target"].mean()},
        {"dataset": "train_inner", "row_count": len(train_inner_df), "pos_count": int(train_inner_df["target"].sum()), "pos_rate": train_inner_df["target"].mean()},
        {"dataset": "valid", "row_count": len(valid_df), "pos_count": int(valid_df["target"].sum()), "pos_rate": valid_df["target"].mean()},
        {"dataset": "official_test", "row_count": len(test_df), "pos_count": int(test_df["target"].sum()), "pos_rate": test_df["target"].mean()},
    ]
)
split_summary

,dataset,row_count,pos_count,pos_rate
0,official_train,60000,1000,0.016667
1,train_inner,48000,800,0.016667
2,valid,12000,200,0.016667
3,official_test,16000,375,0.023438


## 3. 候选模型和缺失处理策略

本轮只比较受控候选项，避免把严谨性增强变成新的大规模调参：

- 模型：Logistic Regression、XGBoost
- 缺失处理策略：`median_all`、`drop_high_missing_median`
- 阈值选择：只在 valid 上完成

In [3]:
results = run_validation_model_selection(
    train_inner_df=train_inner_df,
    valid_df=valid_df,
    test_df=test_df,
    cfg=cfg,
    day6_best_summary_path=cfg.metrics_dir / "day6_best_threshold_summary.csv",
    strategies=["median_all", "drop_high_missing_median"],
    model_names=["logistic_regression_balanced", "xgboost_scale_pos_weight"],
)

validation_threshold_metrics = results["validation_threshold_metrics"]
validation_best_summary = results["validation_best_summary"]
final_test_evaluation = results["final_test_evaluation"]
comparison = results["comparison_with_day6_backtest"]

## 4. valid 上的阈值成本分析结果

下表展示每个模型和缺失处理策略在 valid 上按 total cost 选出的低成本阈值。

In [4]:
validation_best_summary[[
    "selection_rank",
    "model_name",
    "strategy",
    "best_threshold",
    "precision",
    "recall",
    "f2",
    "fp",
    "fn",
    "total_cost",
    "average_precision",
]]

,selection_rank,model_name,strategy,best_threshold,precision,recall,f2,fp,fn,total_cost,average_precision
2,1,xgboost_scale_pos_weight,drop_high_missing_median,0.14,0.331633,0.975,0.702450,393,5,6430,0.867203
3,2,xgboost_scale_pos_weight,median_all,0.16,0.359259,0.970,0.723881,346,6,6460,0.867239
0,3,logistic_regression_balanced,drop_high_missing_median,0.31,0.296178,0.930,0.651261,442,14,11420,0.725343
1,4,logistic_regression_balanced,median_all,0.36,0.312500,0.925,0.664511,407,15,11570,0.722093


## 5. official test 最终评估

这里使用 valid 选出的最佳组合，在 official test 上评估一次。official test 没有参与阈值选择。

In [5]:
final_test_evaluation[[
    "model_name",
    "strategy",
    "threshold",
    "precision",
    "recall",
    "f2",
    "average_precision",
    "fp",
    "fn",
    "total_cost",
    "threshold_source",
]]

,model_name,strategy,threshold,precision,recall,f2,average_precision,fp,fn,total_cost,threshold_source
0,xgboost_scale_pos_weight,drop_high_missing_median,0.14,0.436975,0.970667,0.780111,0.910463,469,11,10190,validation


## 6. 和 Day 6 test 回溯最优结果对比

如果 validation 选择后的 test 成本明显高于 Day 6 test 回溯最优，说明原 Day 6 结果存在一定乐观偏差；如果两者接近，说明当前模型和阈值策略较稳定。

In [6]:
comparison

,comparison_item,model_name,strategy,threshold,precision,recall,f2,fp,fn,total_cost
0,validation_selected_on_test,xgboost_scale_pos_weight,drop_high_missing_median,0.14,0.436975,0.970667,0.780111,469,11,10190
1,day6_test_backtest_best,xgboost_scale_pos_weight,median_all,0.20,0.469231,0.976000,0.802632,414,9,8640
2,delta_valid_minus_day6,None,None,NaN,-0.032256,-0.005333,-0.022520,55,2,1550


## 7. 保存本轮增强输出

输出文件名明确标注 `validation`，不覆盖 Day 4-Day7 旧结果。

In [7]:
cfg.metrics_dir.mkdir(parents=True, exist_ok=True)
cfg.predictions_dir.mkdir(parents=True, exist_ok=True)
cfg.tables_dir.mkdir(parents=True, exist_ok=True)

validation_threshold_metrics.to_csv(cfg.metrics_dir / "validation_threshold_metrics.csv", index=False, encoding="utf-8-sig")
validation_best_summary.to_csv(cfg.metrics_dir / "validation_best_threshold_summary.csv", index=False, encoding="utf-8-sig")
final_test_evaluation.to_csv(cfg.metrics_dir / "final_test_evaluation_from_valid_selection.csv", index=False, encoding="utf-8-sig")
comparison.to_csv(cfg.metrics_dir / "validation_vs_day6_backtest_compare.csv", index=False, encoding="utf-8-sig")
results["validation_predictions"].to_csv(cfg.predictions_dir / "validation_predictions.csv", index=False, encoding="utf-8-sig")
results["final_test_predictions"].to_csv(cfg.predictions_dir / "final_test_predictions_from_valid_selection.csv", index=False, encoding="utf-8-sig")
split_summary.to_csv(cfg.tables_dir / "validation_split_summary.csv", index=False, encoding="utf-8-sig")

## 8. 结论和下一步

本轮增强将阈值选择从 test 回溯分析前移到 valid。若最终 test 成本高于 Day 6 回溯最优，这是合理现象，说明测试集回溯选择阈值存在乐观偏差。

下一步建议做缺失值和特征工程消融实验，例如 `median_with_indicator`、`xgb_native_missing`、`drop_50_missing_median`，但仍应基于 validation 流程选择策略和阈值。